# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaimAli0001/Flyrank-Internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

token = os.getenv("HF_TOKEN")

print("Token loaded:", token is not None)

Token loaded: True


In [2]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{token}'
)
""")

print("DuckDB secret created")

DuckDB secret created


In [3]:
REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

In [4]:
CONTENT_REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
)
"""

content_schema = con.sql(f"""
    DESCRIBE SELECT * FROM {CONTENT_REL}
""").df()

In [5]:
feature_vector = con.sql(f"""
WITH daily_metrics AS (

    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position

    FROM {REL}

    WHERE gsc_data_available IS TRUE
),

half_month_metrics AS (

    SELECT

        client_hash_id,
        content_hash_id,

        -- Monthly features
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,
        AVG(gsc_avg_position) AS march_avg_position,

        -- First half average daily impressions
        AVG(
            CASE
                WHEN report_date <= DATE '2026-03-15'
                THEN gsc_impressions
            END
        ) AS first_half_avg_impressions,

        -- Second half average daily impressions
        AVG(
            CASE
                WHEN report_date > DATE '2026-03-15'
                THEN gsc_impressions
            END
        ) AS second_half_avg_impressions,

        -- First half clicks
        SUM(
            CASE
                WHEN report_date <= DATE '2026-03-15'
                THEN gsc_clicks
                ELSE 0
            END
        ) AS first_half_clicks,

        -- First half impressions
        SUM(
            CASE
                WHEN report_date <= DATE '2026-03-15'
                THEN gsc_impressions
                ELSE 0
            END
        ) AS first_half_impressions,

        -- Second half clicks
        SUM(
            CASE
                WHEN report_date > DATE '2026-03-15'
                THEN gsc_clicks
                ELSE 0
            END
        ) AS second_half_clicks,

        -- Second half impressions
        SUM(
            CASE
                WHEN report_date > DATE '2026-03-15'
                THEN gsc_impressions
                ELSE 0
            END
        ) AS second_half_impressions

    FROM daily_metrics

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT

    client_hash_id,
    content_hash_id,

    march_impressions,

    CASE
        WHEN march_impressions > 0
        THEN march_clicks * 1.0 / march_impressions
        ELSE NULL
    END AS march_ctr,

    march_avg_position,

    second_half_avg_impressions
        - first_half_avg_impressions
        AS impression_trend,

    CASE
        WHEN second_half_impressions > 0
         AND first_half_impressions > 0
        THEN
            (second_half_clicks * 1.0 / second_half_impressions)
            -
            (first_half_clicks * 1.0 / first_half_impressions)
        ELSE NULL
    END AS ctr_trend

FROM half_month_metrics

""").df()

feature_vector

,client_hash_id,content_hash_id,march_impressions,march_ctr,march_avg_position,impression_trend,ctr_trend
0,client_ff644d8251367cbb,content_004f588c22238f5f,84.0,0.000000,52.230797,3.674603,0.000000
1,client_ff644d8251367cbb,content_42b4dc03cd5a1e2d,833.0,0.001200,27.452774,-10.583333,-0.002062
2,client_ff644d8251367cbb,content_dd87654a316c77e3,51.0,0.000000,6.125000,0.272727,0.000000
3,client_ff644d8251367cbb,content_4aa9c6a5f1c5190a,513.0,0.007797,13.197906,0.029167,-0.008323
4,client_ff644d8251367cbb,content_405eba89b1fea3b8,96.0,0.000000,6.824968,0.410714,0.000000
...,...,...,...,...,...,...,...
176733,client_0fa64a184f18a4a0,content_f1491352edcfdbf9,2.0,0.000000,4.000000,NaN,NaN
176734,client_b77d0d5f08f05e64,content_5a6edf83ecbc57ad,1.0,0.000000,4.000000,NaN,NaN
176735,client_86ebc2f12c01f586,content_330e2738425b0db2,61.0,0.000000,11.032787,NaN,NaN
176736,client_08a6a72ff48e62c0,content_4e5fa14b358fbca8,1.0,0.000000,45.000000,NaN,NaN


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_vector.shape



(176738, 7)

In [7]:

feature_vector.head()


,client_hash_id,content_hash_id,march_impressions,march_ctr,march_avg_position,impression_trend,ctr_trend
0,client_ff644d8251367cbb,content_004f588c22238f5f,84.0,0.000000,52.230797,3.674603,0.000000
1,client_ff644d8251367cbb,content_42b4dc03cd5a1e2d,833.0,0.001200,27.452774,-10.583333,-0.002062
2,client_ff644d8251367cbb,content_dd87654a316c77e3,51.0,0.000000,6.125000,0.272727,0.000000
3,client_ff644d8251367cbb,content_4aa9c6a5f1c5190a,513.0,0.007797,13.197906,0.029167,-0.008323
4,client_ff644d8251367cbb,content_405eba89b1fea3b8,96.0,0.000000,6.824968,0.410714,0.000000


In [8]:

feature_vector.describe()

,march_impressions,march_ctr,march_avg_position,impression_trend,ctr_trend
count,176738.000000,176738.000000,176738.000000,141467.000000,141467.000000
mean,1587.986675,0.004594,15.999277,3.351631,-0.000480
std,5431.337724,0.037760,17.686260,116.640447,0.039182
min,1.000000,0.000000,0.000000,-5580.925000,-1.000000
25%,20.000000,0.000000,5.001970,-3.029167,-0.000142
50%,173.000000,0.000000,8.505296,0.000000,0.000000
75%,1039.000000,0.002158,20.369190,3.562500,0.000000
max,617124.000000,1.000000,309.000000,16042.966346,1.000000


In [9]:
position_signal = con.sql("""
SELECT

    CASE

        WHEN march_avg_position BETWEEN 1 AND 3
            THEN '1-3'

        WHEN march_avg_position BETWEEN 4 AND 7
            THEN '4-7'

        WHEN march_avg_position BETWEEN 8 AND 12
            THEN '8-12'

        WHEN march_avg_position BETWEEN 13 AND 20
            THEN '13-20'

        ELSE '21+'

    END AS position_bucket,

    COUNT(*) AS n,

    AVG(march_ctr) AS avg_ctr

FROM feature_vector

GROUP BY position_bucket

ORDER BY
CASE position_bucket
    WHEN '1-3' THEN 1
    WHEN '4-7' THEN 2
    WHEN '8-12' THEN 3
    WHEN '13-20' THEN 4
    ELSE 5
END

""").df()

position_signal

,position_bucket,n,avg_ctr
0,1-3,15105,0.010595
1,4-7,43217,0.005293
2,8-12,26885,0.003578
3,13-20,19312,0.003277
4,21+,72219,0.003651


### Signal 1 – CTR vs Search Position

**Belief**

Pages ranking higher in search results should generally achieve a higher click-through rate (CTR) than pages ranking lower.

**Result**

Pages were grouped into five position buckets (1–3, 4–7, 8–12, 13–20 and 21+). The average CTR decreased steadily from the top-ranking pages to the lower-ranking buckets. Although the 21+ bucket shows a small increase compared with the 13–20 bucket, it combines a very wide range of positions and therefore contains greater variation.

**Verdict**

**CONFIRMED**

The overall relationship between search position and CTR matches the expected search behaviour. This confirms that average position is an informative feature for identifying pages that may require snippet review.

In [10]:
impression_distribution = con.sql("""
SELECT
    MIN(march_impressions) AS minimum,
    quantile_cont(march_impressions, 0.25) AS q1,
    MEDIAN(march_impressions) AS median,
    quantile_cont(march_impressions, 0.75) AS q3,
    MAX(march_impressions) AS maximum,
    AVG(march_impressions) AS average
FROM feature_vector
""").df()

impression_distribution

,minimum,q1,median,q3,maximum,average
0,1.0,20.0,173.0,1039.0,617124.0,1587.986675


In [11]:
impression_signal = con.sql("""
SELECT

    CASE

        WHEN march_impressions BETWEEN 1 AND 20
            THEN '1-20'

        WHEN march_impressions BETWEEN 21 AND 173
            THEN '21-173'

        WHEN march_impressions BETWEEN 174 AND 1039
            THEN '174-1039'

        WHEN march_impressions BETWEEN 1040 AND 5000
            THEN '1040-5000'

        ELSE '5001+'

    END AS impression_bucket,

    COUNT(*) AS n,

    AVG(march_ctr) AS avg_ctr

FROM feature_vector

GROUP BY impression_bucket

ORDER BY
CASE impression_bucket
    WHEN '1-20' THEN 1
    WHEN '21-173' THEN 2
    WHEN '174-1039' THEN 3
    WHEN '1040-5000' THEN 4
    ELSE 5
END

""").df()

impression_signal

,impression_bucket,n,avg_ctr
0,1-20,44983,0.010102
1,21-173,43409,0.002820
2,174-1039,44186,0.002371
3,1040-5000,30870,0.002961
4,5001+,13290,0.002929


### Signal 2 – Business Opportunity (Search Volume)

**Belief**

Pages with higher search visibility represent greater business opportunity because improving their performance can generate more additional clicks.

**Result**

Pages were grouped into five impression buckets based on the observed distribution. The lowest-impression bucket showed a noticeably higher average CTR, while the remaining buckets had similar CTR values with no strong monotonic trend. This suggests that CTR alone is not strongly determined by the number of impressions.

**Verdict**

**MIXED**

The data does not show a strong relationship between impressions and CTR. However, impression volume remains an important prioritization signal because improvements on high-impression pages have greater potential business impact, even when their average CTR is similar to other groups.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Baseline Rule

The baseline rule ranks pages for snippet review using three observable search signals.

1. **Business Opportunity** – Pages with more impressions receive a higher score because improving their CTR can generate more additional clicks.

2. **Snippet Opportunity** – Pages with lower CTR receive a higher score because they may benefit from improved titles and meta descriptions.

3. **Position Opportunity** – Pages ranking between positions 4 and 12 receive the highest score because they are already visible in search results and small improvements may increase clicks.

The final priority score is the sum of the three component scores. Higher scores indicate higher priority for review.

**Reason Code**

`LOW_CTR_HIGH_VISIBILITY`

**Action Label**

`SNIPPET_REVIEW`

In [16]:
baseline_queue = con.sql("""
SELECT

    *,

    ---------------------------------------------------
    -- Impression Score
    ---------------------------------------------------
    CASE

        WHEN march_impressions BETWEEN 1 AND 20 THEN 1
        WHEN march_impressions BETWEEN 21 AND 173 THEN 2
        WHEN march_impressions BETWEEN 174 AND 1039 THEN 3
        WHEN march_impressions BETWEEN 1040 AND 5000 THEN 4
        ELSE 5

    END AS impression_score,

    ---------------------------------------------------
    -- CTR Score
    ---------------------------------------------------
    CASE

        WHEN march_ctr = 0 THEN 5
        WHEN march_ctr <= 0.002 THEN 4
        WHEN march_ctr <= 0.005 THEN 3
        WHEN march_ctr <= 0.01 THEN 2
        ELSE 1

    END AS ctr_score,

    ---------------------------------------------------
    -- Position Score
    ---------------------------------------------------
    CASE

        WHEN march_avg_position BETWEEN 4 AND 7 THEN 5
        WHEN march_avg_position BETWEEN 8 AND 12 THEN 4
        WHEN march_avg_position BETWEEN 13 AND 20 THEN 3
        WHEN march_avg_position > 20 THEN 2
        ELSE 1

    END AS position_score

FROM feature_vector

""").df()

In [28]:
baseline_queue["priority_score"] = (
    baseline_queue["impression_score"]
    + baseline_queue["ctr_score"]
    + baseline_queue["position_score"]
)

baseline_queue["reason_code"] = "LOW_CTR_HIGH_VISIBILITY"
baseline_queue["action_label"] = "SNIPPET_REVIEW"

In [19]:
baseline_queue = baseline_queue.sort_values(
    by=[
        "priority_score",
        "march_impressions",
        "march_ctr"
    ],
    ascending=[
        False,   # Higher priority first
        False,   # Higher impressions first
        True     # Lower CTR first
    ]
).reset_index(drop=True)

baseline_queue.head(20)

,client_hash_id,content_hash_id,march_impressions,march_ctr,march_avg_position,impression_trend,ctr_trend,impression_score,ctr_score,position_score,priority_score,reason_code,action_label
0,client_62f4a7e64f5e0096,content_0c5606abaaab3178,38865.0,0.0,5.694764,-1150.791667,0.0,5,5,5,15,LOW_CTR_HIGH_VISIBILITY,SNIPPET_REVIEW
1,client_e547b89c05043229,content_713b157e9c77690a,24908.0,0.0,4.009764,680.475962,0.0,5,5,5,15,LOW_CTR_HIGH_VISIBILITY,SNIPPET_REVIEW
2,client_e547b89c05043229,content_0e2e4d3ab02abc1a,11187.0,0.0,4.376694,479.317308,0.0,5,5,5,15,LOW_CTR_HIGH_VISIBILITY,SNIPPET_REVIEW
3,client_e547b89c05043229,content_f5a7a2559d483a54,10886.0,0.0,4.204145,149.312500,0.0,5,5,5,15,LOW_CTR_HIGH_VISIBILITY,SNIPPET_REVIEW
4,client_73cda7b4e4f265ea,content_520e203a08cd69ee,10462.0,0.0,4.259897,622.020833,0.0,5,5,5,15,LOW_CTR_HIGH_VISIBILITY,SNIPPET_REVIEW
5,client_62f4a7e64f5e0096,content_e9bec3c326f675f3,9312.0,0.0,5.766627,180.420833,0.0,5,5,5,15,LOW_CTR_HIGH_VISIBILITY,SNIPPET_REVIEW
6,client_20259bd6705d81d4,content_7015cdffa6cf2bcf,9218.0,0.0,5.464271,193.145833,0.0,5,5,5,15,LOW_CTR_HIGH_VISIBILITY,SNIPPET_REVIEW
7,client_62f4a7e64f5e0096,content_b5409712af074f7b,8608.0,0.0,4.429576,204.562500,0.0,5,5,5,15,LOW_CTR_HIGH_VISIBILITY,SNIPPET_REVIEW
8,client_73cda7b4e4f265ea,content_92d2c3fc8bf462c9,8343.0,0.0,6.503199,-156.041667,0.0,5,5,5,15,LOW_CTR_HIGH_VISIBILITY,SNIPPET_REVIEW
9,client_62f4a7e64f5e0096,content_a6686fbc7311edad,8258.0,0.0,5.320302,350.791667,0.0,5,5,5,15,LOW_CTR_HIGH_VISIBILITY,SNIPPET_REVIEW


In [33]:
import os

os.makedirs("../outputs", exist_ok=True)

baseline_queue.to_csv(
    "../outputs/baseline_action_score.csv",
    index=False
)

print("baseline_action_score.csv created successfully.")

baseline_action_score.csv created successfully.


In [21]:
baseline_queue.head(10)

,client_hash_id,content_hash_id,march_impressions,march_ctr,march_avg_position,impression_trend,ctr_trend,impression_score,ctr_score,position_score,priority_score,reason_code,action_label
0,client_62f4a7e64f5e0096,content_0c5606abaaab3178,38865.0,0.0,5.694764,-1150.791667,0.0,5,5,5,15,LOW_CTR_HIGH_VISIBILITY,SNIPPET_REVIEW
1,client_e547b89c05043229,content_713b157e9c77690a,24908.0,0.0,4.009764,680.475962,0.0,5,5,5,15,LOW_CTR_HIGH_VISIBILITY,SNIPPET_REVIEW
2,client_e547b89c05043229,content_0e2e4d3ab02abc1a,11187.0,0.0,4.376694,479.317308,0.0,5,5,5,15,LOW_CTR_HIGH_VISIBILITY,SNIPPET_REVIEW
3,client_e547b89c05043229,content_f5a7a2559d483a54,10886.0,0.0,4.204145,149.312500,0.0,5,5,5,15,LOW_CTR_HIGH_VISIBILITY,SNIPPET_REVIEW
4,client_73cda7b4e4f265ea,content_520e203a08cd69ee,10462.0,0.0,4.259897,622.020833,0.0,5,5,5,15,LOW_CTR_HIGH_VISIBILITY,SNIPPET_REVIEW
5,client_62f4a7e64f5e0096,content_e9bec3c326f675f3,9312.0,0.0,5.766627,180.420833,0.0,5,5,5,15,LOW_CTR_HIGH_VISIBILITY,SNIPPET_REVIEW
6,client_20259bd6705d81d4,content_7015cdffa6cf2bcf,9218.0,0.0,5.464271,193.145833,0.0,5,5,5,15,LOW_CTR_HIGH_VISIBILITY,SNIPPET_REVIEW
7,client_62f4a7e64f5e0096,content_b5409712af074f7b,8608.0,0.0,4.429576,204.562500,0.0,5,5,5,15,LOW_CTR_HIGH_VISIBILITY,SNIPPET_REVIEW
8,client_73cda7b4e4f265ea,content_92d2c3fc8bf462c9,8343.0,0.0,6.503199,-156.041667,0.0,5,5,5,15,LOW_CTR_HIGH_VISIBILITY,SNIPPET_REVIEW
9,client_62f4a7e64f5e0096,content_a6686fbc7311edad,8258.0,0.0,5.320302,350.791667,0.0,5,5,5,15,LOW_CTR_HIGH_VISIBILITY,SNIPPET_REVIEW


In [24]:
baseline_queue["priority_score"].describe()

count    176738.000000
mean          9.647461
std           1.954337
min           3.000000
25%           8.000000
50%          10.000000
75%          11.000000
max          15.000000
Name: priority_score, dtype: float64

The ranked queue is created by combining three business-driven scores:

- **Impression Score:** Prioritizes pages with greater search visibility.
- **CTR Score:** Prioritizes pages with lower click-through rates.
- **Position Score:** Prioritizes pages ranking between positions 4 and 12, where snippet improvements are expected to have the greatest impact.

The final priority score is the sum of these three component scores. Pages are ranked by priority score, with ties broken by higher impressions and lower CTR. Each page receives the reason code `LOW_CTR_HIGH_VISIBILITY` and the action label `SNIPPET_REVIEW`.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [29]:
top20_review = baseline_queue.head(20).copy()

# Confidence
top20_review["confidence"] = "Low"

top20_review.loc[
    (top20_review["priority_score"] == 15) &
    (top20_review["impression_trend"] <= 0),
    "confidence"
] = "High"

top20_review.loc[
    (top20_review["priority_score"] == 15) &
    (top20_review["impression_trend"] > 0),
    "confidence"
] = "Medium-High"

top20_review.loc[
    top20_review["priority_score"].between(13, 14),
    "confidence"
] = "Medium-High"

top20_review.loc[
    top20_review["priority_score"].between(11, 12),
    "confidence"
] = "Medium"

# What would make it wrong
top20_review["what_would_make_it_wrong"] = (
    "Manual review should confirm that the snippet is the primary issue."
)

top20_review.loc[
    top20_review["march_ctr"] == 0,
    "what_would_make_it_wrong"
] = (
    "The page may receive impressions for irrelevant queries or be affected by SERP features rather than the snippet."
)

top20_review.loc[
    top20_review["impression_trend"] < 0,
    "what_would_make_it_wrong"
] = (
    "The traffic decline may be caused by seasonality, algorithm updates, or competitor activity rather than the snippet."
)

top20_review.loc[
    top20_review["impression_trend"] > 0,
    "what_would_make_it_wrong"
] = (
    "The page is already gaining visibility, so monitoring may be more appropriate before making changes."
)
top20_review[
    [
        "content_hash_id",
        "priority_score",
        "action_label",
        "reason_code",
        "confidence",
        "what_would_make_it_wrong"
    ]
]

,content_hash_id,priority_score,action_label,reason_code,confidence,what_would_make_it_wrong
0,content_0c5606abaaab3178,15,SNIPPET_REVIEW,LOW_CTR_HIGH_VISIBILITY,High,The traffic decline may be caused by seasonali...
1,content_713b157e9c77690a,15,SNIPPET_REVIEW,LOW_CTR_HIGH_VISIBILITY,Medium-High,"The page is already gaining visibility, so mon..."
2,content_0e2e4d3ab02abc1a,15,SNIPPET_REVIEW,LOW_CTR_HIGH_VISIBILITY,Medium-High,"The page is already gaining visibility, so mon..."
3,content_f5a7a2559d483a54,15,SNIPPET_REVIEW,LOW_CTR_HIGH_VISIBILITY,Medium-High,"The page is already gaining visibility, so mon..."
4,content_520e203a08cd69ee,15,SNIPPET_REVIEW,LOW_CTR_HIGH_VISIBILITY,Medium-High,"The page is already gaining visibility, so mon..."
5,content_e9bec3c326f675f3,15,SNIPPET_REVIEW,LOW_CTR_HIGH_VISIBILITY,Medium-High,"The page is already gaining visibility, so mon..."
6,content_7015cdffa6cf2bcf,15,SNIPPET_REVIEW,LOW_CTR_HIGH_VISIBILITY,Medium-High,"The page is already gaining visibility, so mon..."
7,content_b5409712af074f7b,15,SNIPPET_REVIEW,LOW_CTR_HIGH_VISIBILITY,Medium-High,"The page is already gaining visibility, so mon..."
8,content_92d2c3fc8bf462c9,15,SNIPPET_REVIEW,LOW_CTR_HIGH_VISIBILITY,High,The traffic decline may be caused by seasonali...
9,content_a6686fbc7311edad,15,SNIPPET_REVIEW,LOW_CTR_HIGH_VISIBILITY,Medium-High,"The page is already gaining visibility, so mon..."


## 3. Top-20 Review

The Top-20 pages were reviewed using the baseline ranking produced in Section 2. Instead of writing manual comments for each page, a small rule-based review engine was used to generate consistent recommendations.

Each selected page receives:

- **Action Label:** `SNIPPET_REVIEW`
- **Reason Code:** `LOW_CTR_HIGH_VISIBILITY`

The review engine also assigns a confidence level and a caution note using observable search signals rather than manual judgement.

### Confidence Rules

- **High:** Maximum priority score (15) with a stable or declining impression trend.
- **Medium-High:** Maximum priority score with a positive impression trend, or priority scores between 13 and 14.
- **Medium:** Priority scores between 11 and 12.
- **Low:** Remaining pages with weaker evidence.

### Review Cautions

The recommendation may not always be correct. Possible reasons include:

- The page appears in search results for queries with naturally low click-through rates.
- Rich search features (featured snippets, AI Overviews, knowledge panels, etc.) reduce organic clicks.
- Changes in traffic may be caused by seasonality, competitor activity, or search algorithm updates rather than the page snippet.
- Pages that are already gaining visibility may benefit from monitoring before making changes.

This review process keeps the baseline transparent, repeatable, and scalable while acknowledging that final decisions should still involve human review.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## 4. Weak Picks + Leakage Check

### Weak Picks

The baseline rule is intentionally simple and may produce false positives in some situations.

Examples include:

- Pages with high impressions but naturally low click-through rates because of search intent.
- Pages affected by rich search features (such as AI Overviews, featured snippets, or knowledge panels) where users do not need to click.
- Pages that are already gaining impressions, where monitoring may be more appropriate than immediate snippet changes.

These cases highlight where human review remains important before taking action.

### Leakage Check

The baseline was reviewed for data leakage.

Confirmed:

- Only March 2026 data was used to construct the feature vector.
- No future observations or future windows were used.
- No label-derived features were included.
- No product flags or outcome variables were used in the scoring rule.
- Every feature is available before the recommendation is made.

The baseline is therefore suitable as a transparent rule-based benchmark for Week 5.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.